In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 22
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


## UNODC Homicide Pipeline

**Source:** UNODC Homicide Statistics via Our World in Data
**Access:** Automated OWID CSV — no registration required
**Download instructions:** See `docs/instructions_data_maintenance.md` — UNODC_HOMICIDE section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Homicide rate (per 100,000) | Personal security and order | Primary tier 1 |

In [2]:
import requests
import io
import pandas as pd
from datetime import datetime

# UNODC Homicide Statistics via Our World in Data
# Same stable URL pattern as TI CPI
OWID_UNODC_URL = "https://ourworldindata.org/grapher/homicide-rate-unodc.csv?v=1&csvType=full&useColumnShortNames=false"

print("Downloading UNODC homicide data from Our World in Data...")
response = requests.get(OWID_UNODC_URL, timeout=30)
print(f"Status: {response.status_code}, Size: {len(response.content)/1024:.1f}KB")

unodc_raw = pd.read_csv(io.StringIO(response.text))
print(f"\nShape: {unodc_raw.shape}")
print(f"Columns: {list(unodc_raw.columns)}")
print(f"Years: {unodc_raw['Year'].min()} — {unodc_raw['Year'].max()}")
print(f"Countries: {unodc_raw['Code'].nunique()}")
print(unodc_raw.head(3))

Status: 200, Size: 175.6KB

Shape: (4885, 5)
Columns: ['Entity', 'Code', 'Year', 'Homicide rate per 100,000 population', 'World region according to OWID']
Years: 1990 — 2024
Countries: 208
        Entity Code  Year  Homicide rate per 100,000 population  \
0  Afghanistan  AFG  2009                              4.059550   
1  Afghanistan  AFG  2010                              3.475452   
2  Afghanistan  AFG  2011                              4.194535   

  World region according to OWID  
0                           Asia  
1                           Asia  
2                           Asia  


In [3]:
# Filter and rename columns
unodc = unodc_raw.copy()
unodc = unodc.rename(columns={
    'Entity':                                'country_name',
    'Code':                                  'country_code',
    'Year':                                  'year',
    'Homicide rate per 100,000 population':  'unodc_homicide_rate',
})

# Drop OWID region column
unodc = unodc.drop(columns=['World region according to OWID'])

# Drop rows with no country code — regional aggregates
unodc = unodc[unodc['country_code'].notna() & (unodc['country_code'] != '')].copy()

# Filter to framework start year
unodc = unodc[unodc['year'] >= FRAMEWORK_START_YEAR].copy()
unodc = unodc.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {unodc.shape}")
print(f"Years: {unodc['year'].min()} — {unodc['year'].max()}")
print(f"Countries: {unodc['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (unodc.isnull().sum() / len(unodc) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(unodc.head(3))

Shape: (4388, 4)
Years: 1990 — 2024
Countries: 208

Missing values (%):
Series([], dtype: float64)
  country_name country_code  year  unodc_homicide_rate
0  Afghanistan          AFG  2009             4.059550
1  Afghanistan          AFG  2010             3.475452
2  Afghanistan          AFG  2011             4.194535


In [4]:
# Derive metadata from data — no hardcoding
latest_year = str(int(unodc['year'].max()))
data_as_of_date = latest_year

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "unodc_clean.csv")
unodc.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {unodc.shape}")

# Update download log
update_entry(
    "UNODC_HOMICIDE",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="unodc_clean.csv",
    latest_available_version=latest_year,
    notes="Homicide rate per 100,000. Downloaded via OWID historical panel. Coverage: 1990-2024, 208 countries."
)

print_entry("UNODC_HOMICIDE")

Written: /Users/boulanger/Documents/governance-framework/data/processed/unodc_clean.csv
Shape: (4388, 4)
[download_log] Updated entry for UNODC_HOMICIDE
  source_id: UNODC_HOMICIDE
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-06-12
  data_as_of_date: 2024
  local_filename: unodc_clean.csv
  latest_available_version: 2024
  no_update_reason: nan
  notes: Homicide rate per 100,000. Downloaded via OWID historical panel. Coverage: 1990-2024, 208 countries.
